You can download the `requirements.txt` for this course from the workspace of this lab. `File --> Open...`

# L2: Create Agents to Research and Write an Article

In this lesson, you will be introduced to the foundational concepts of multi-agent systems and get an overview of the crewAI framework.

The libraries are already installed in the classroom. If you're running this notebook on your own machine, you can install the following:
```Python
!pip install crewai==0.28.8 crewai_tools==0.1.6 langchain_community==0.0.29
```

In [2]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

- Import from the crewAI libray.

In [3]:
from crewai import Agent, Task, Crew

- As a LLM for your agents, you'll be using OpenAI's `gpt-3.5-turbo`.

**Optional Note:** crewAI also allow other popular models to be used as a LLM for your Agents. You can see some of the examples at the [bottom of the notebook](#1).

In [ ]:
import os
from com.example.agentic.utils import get_openai_api_key

openai_api_key = get_openai_api_key()
os.environ["OPENAI_MODEL_NAME"] = 'llama3.2:1b-instruct-q8_0'
#os.environ["OPENAI_MODEL_NAME"] = 'gpt-3.5-turbo'

from crewai import LLM
llm = LLM(model="ollama/llama3.2:1b-instruct-q8_0", base_url="http://localhost:11434")

## Creating Agents

- Define your Agents, and provide them a `role`, `goal` and `backstory`.
- It has been seen that LLMs perform better when they are role playing.

### Agent: Planner

**Note**: The benefit of using _multiple strings_ :
```Python
varname = "line 1 of text"
          "line 2 of text"
```

versus the _triple quote docstring_:
```Python
varname = """line 1 of text
             line 2 of text
          """
```
is that it can avoid adding those whitespaces and newline characters, making it better formatted to be passed to the LLM.

In [4]:
planner = Agent(
    role="Content Planner",
    goal="Plan engaging and factually accurate content on {topic}",
    backstory="You're working on planning a blog article "
              "about the topic: {topic}."
              "You collect information that helps the "
              "audience learn something "
              "and make informed decisions. "
              "Your work is the basis for "
              "the Content Writer to write an article on this topic.",
    allow_delegation=False,
	verbose=True
)

### Agent: Writer

In [5]:
writer = Agent(
    role="Content Writer",
    goal="Write insightful and factually accurate "
         "opinion piece about the topic: {topic}",
    backstory="You're working on a writing "
              "a new opinion piece about the topic: {topic}. "
              "You base your writing on the work of "
              "the Content Planner, who provides an outline "
              "and relevant context about the topic. "
              "You follow the main objectives and "
              "direction of the outline, "
              "as provide by the Content Planner. "
              "You also provide objective and impartial insights "
              "and back them up with information "
              "provide by the Content Planner. "
              "You acknowledge in your opinion piece "
              "when your statements are opinions "
              "as opposed to objective statements.",
    allow_delegation=False,
    verbose=True
)

### Agent: Editor

In [6]:
editor = Agent(
    role="Editor",
    goal="Edit a given blog post to align with "
         "the writing style of the organization. ",
    backstory="You are an editor who receives a blog post "
              "from the Content Writer. "
              "Your goal is to review the blog post "
              "to ensure that it follows journalistic best practices,"
              "provides balanced viewpoints "
              "when providing opinions or assertions, "
              "and also avoids major controversial topics "
              "or opinions when possible.",
    allow_delegation=False,
    verbose=True
)

## Creating Tasks

- Define your Tasks, and provide them a `description`, `expected_output` and `agent`.

### Task: Plan

In [7]:
plan = Task(
    description=(
        "1. Prioritize the latest trends, key players, "
            "and noteworthy news on {topic}.\n"
        "2. Identify the target audience, considering "
            "their interests and pain points.\n"
        "3. Develop a detailed content outline including "
            "an introduction, key points, and a call to action.\n"
        "4. Include SEO keywords and relevant data or sources."
    ),
    expected_output="A comprehensive content plan document "
        "with an outline, audience analysis, "
        "SEO keywords, and resources.",
    agent=planner,
)

### Task: Write

In [8]:
write = Task(
    description=(
        "1. Use the content plan to craft a compelling "
            "blog post on {topic}.\n"
        "2. Incorporate SEO keywords naturally.\n"
		"3. Sections/Subtitles are properly named "
            "in an engaging manner.\n"
        "4. Ensure the post is structured with an "
            "engaging introduction, insightful body, "
            "and a summarizing conclusion.\n"
        "5. Proofread for grammatical errors and "
            "alignment with the brand's voice.\n"
    ),
    expected_output="A well-written blog post "
        "in markdown format, ready for publication, "
        "each section should have 2 or 3 paragraphs.",
    agent=writer,
)

### Task: Edit

In [9]:
edit = Task(
    description=("Proofread the given blog post for "
                 "grammatical errors and "
                 "alignment with the brand's voice."),
    expected_output="A well-written blog post in markdown format, "
                    "ready for publication, "
                    "each section should have 2 or 3 paragraphs.",
    agent=editor
)

## Creating the Crew

- Create your crew of Agents
- Pass the tasks to be performed by those agents.
    - **Note**: *For this simple example*, the tasks will be performed sequentially (i.e they are dependent on each other), so the _order_ of the task in the list _matters_.
- `verbose=2` allows you to see all the logs of the execution. 

In [10]:
crew = Crew(
    agents=[planner, writer, editor],
    tasks=[plan, write, edit],
    verbose=True
)

## Running the Crew

**Note**: LLMs can provide different outputs for they same input, so what you get might be different than what you see in the video.

In [ ]:
result = crew.kickoff(inputs={"topic": "Artificial Intelligence"})

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 5b4977c6-c4cd-44e9-9c9b-c623449a00ea                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: 1. Prioritize the latest trends, key players, and noteworthy news on Artificial Intelligence.            │
│  2. Identify the target audience, considering their interests and pain points.                                  │
│  3. Develop a detailed content outline including an introduction, key points, and a call to action.             │
│  4. Include SEO keywords and relevant data or sources.                                                          │
│  ID: ab202900-2e67-4308-9196-4f8b76590c7c                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Content Planner                                                                                         │
│                                                                                                                 │
│  Task: 1. Prioritize the latest trends, key players, and noteworthy news on Artificial Intelligence.            │
│  2. Identify the target audience, considering their interests and pain points.                                  │
│  3. Develop a detailed content outline including an introduction, key points, and a call to action.             │
│  4. Include SEO keywords and relevant data or sources.                                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Content Planner                                                                                         │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Comprehensive Content Plan Document**                                                                        │
│                                                                                                                 │
│  **Document Title:** Emerging Trends, Key Players, and Noteworthy News in Artificial Intelligence (AI)          │
│                                                                                                                 │
│  **I. Introduction**                                                                                            │
│                                                                                                                 │
│  * Brief overview of Artificial Intelligence (AI) and its significance                                          │
│  * Importance of staying updated on the latest trends, key players, and noteworthy news in AI                   │
│  * Thesis statement: This content plan explores the current state of AI, highlighting key trends, influential   │
│  companies, and significant announcements.                                                                      │
│                                                                                                                 │
│  **II. Target Audience**                                                                                        │
│                                                                                                                 │
│  * **Demographics:** Blog readers who are interested in technology, innovation, and artificial intelligence.    │
│  * **Psychographics:** Professionals, enthusiasts, and curious individuals who want to understand the latest    │
│  developments and technologies related to AI.                                                                   │
│  * **Pain points:**                                                                                             │
│          + Lack of clarity on how AI drives business decisions                                                  │
│          + Difficulty understanding complex AI concepts and applications                                        │
│          + Wanting real-time insights into the latest industry trends and innovations                           │
│                                                                                                                 │
│  **III. Key Topics**                                                                                            │
│                                                                                                                 │
│  1. **Natural Language Processing (NLP):** Emerging trend, challenges, and successes.                           │
│  2. **Machine Learning (ML):** Latest breakthroughs, applications, and key players.                             │
│  3. **Edge Computing:** Rise of edge computing in AI adoption and its benefits.                                 │
│  4. **Quantum Computing and AI:** Potential intersection and future implications.                               │
│                                                                                                                 │
│  **IV. Noteworthy News and Incidents**                                                                          │
│                                                        

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 1. Prioritize the latest trends, key players, and noteworthy news on Artificial Intelligence.            │
│  2. Identify the target audience, considering their interests and pain points.                                  │
│  3. Develop a detailed content outline including an introduction, key points, and a call to action.             │
│  4. Include SEO keywords and relevant data or sources.                                                          │
│  Agent: Content Planner                                                                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Editor                                                                                                  │
│                                                                                                                 │
│  Task: Proofread the given blog post for grammatical errors and alignment with the brand's voice.               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Editor                                                                                                  │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Comprehensive Content Plan Document**                                                                        │
│                                                                                                                 │
│  **Document Title:** Emerging Trends, Key Players, and Noteworthy News in Artificial Intelligence (AI)          │
│                                                                                                                 │
│  **I. Introduction**                                                                                            │
│  ===============                                                                                                │
│                                                                                                                 │
│  Artificial Intelligence (AI) refers to the development of computer systems that can perform tasks that         │
│  typically require human intelligence. From recognizing images to understanding natural language, AI has been   │
│  transforming various industries and aspects of our lives. As the field continues to advance, it's essential    │
│  to stay updated on its current state, exploring key trends, influential companies, and significant             │
│  announcements.                                                                                                 │
│                                                                                                                 │
│  In this content plan, we will delve into the world of Artificial Intelligence, discussing its significance,    │
│  importance, and recent developments. We'll examine not only the technical aspects but also the broader         │
│  implications of AI on our profession, society, and business landscape. Finally, we aim to provide readers      │
│  with a comprehensive understanding of the subject, highlighting key trends, upcoming news, and further         │
│  resources for those interested in exploring this rapidly evolving field.                                       │
│                                                                                                                 │
│  ## **Noteworthy News: DeepMind AI Acquisition by Alphabet**                                                    │
│                                                                                                                 │
│  DeepMind is a British artificial intelligence company founded in 2010. It's part of the Google subsidiary but  │
│  remains an independent entity offering its technological innovations under the brand name DeepMind AI.         │
│                                                                                                                 │
│  DeepMind was acquired by Alphabet, an American multinational conglomerate that oversees the parent companies   │
│  of YouTube and Google Search, on [Date]. This acquisition marked significant breakthroughs in artificial       │
│  intelligence, highlighting potential applications such as computer vision, natural language processing, and    │
│  machine learning.                                                                                              │
│                                                                                                                 │
│  ### **Acquisition Insights: Breaking Down Complexity t

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Proofread the given blog post for grammatical errors and alignment with the brand's voice.               │
│  Agent: Editor                                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────── Trace Batch Finalization ──────────────────────────╮
│ ✅ Trace batch finalized with session ID:                                    │
│ 462cf7d1-b19b-4ce1-8f00-281540513dee                                         │
│                                                                              │
│ 🔗 View here:                                                                │
│ https://app.crewai.com/crewai_plus/ephemeral_trace_batches/462cf7d1-b19b-4ce │
│ 1-8f00-281540513dee?access_code=TRACE-39fd8b2fa9                             │
│ 🔑 Access Code: TRACE-39fd8b2fa9                                             │
╰──────────────────────────────────────────────────────────────────────────────╯


╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 5b4977c6-c4cd-44e9-9c9b-c623449a00ea                                                                       │
│  Final Output: **Comprehensive Content Plan Document**                                                          │
│                                                                                                                 │
│  **Document Title:** Emerging Trends, Key Players, and Noteworthy News in Artificial Intelligence (AI)          │
│                                                                                                                 │
│  **I. Introduction**                                                                                            │
│  ===============                                                                                                │
│                                                                                                                 │
│  Artificial Intelligence (AI) refers to the development of computer systems that can perform tasks that         │
│  typically require human intelligence. From recognizing images to understanding natural language, AI has been   │
│  transforming various industries and aspects of our lives. As the field continues to advance, it's essential    │
│  to stay updated on its current state, exploring key trends, influential companies, and significant             │
│  announcements.                                                                                                 │
│                                                                                                                 │
│  In this content plan, we will delve into the world of Artificial Intelligence, discussing its significance,    │
│  importance, and recent developments. We'll examine not only the technical aspects but also the broader         │
│  implications of AI on our profession, society, and business landscape. Finally, we aim to provide readers      │
│  with a comprehensive understanding of the subject, highlighting key trends, upcoming news, and further         │
│  resources for those interested in exploring this rapidly evolving field.                                       │
│                                                                                                                 │
│  ## **Noteworthy News: DeepMind AI Acquisition by Alphabet**                                                    │
│                                                                                                                 │
│  DeepMind is a British artificial intelligence company founded in 2010. It's part of the Google subsidiary but  │
│  remains an independent entity offering its technological innovations under the brand name DeepMind AI.         │
│                                                                                                                 │
│  DeepMind was acquired by Alphabet, an American multinational conglomerate that oversees the parent companies   │
│  of YouTube and Google Search, on [Date]. This acquisition marked significant breakthroughs in artificial       │
│  intelligence, highlighting potential applications such as computer vision, natural language processing, and    │
│  machine learning.                                                                                              │
│                                                                                                                 │
│  ### **Acquisition Insights: Breaking Down Complexity 

- Display the results of your execution as markdown in the notebook.

In [13]:
from IPython.display import Markdown
Markdown(result.raw)

Here is the final answer in the requested format:

# Current Task: Proofread the given blog post for grammatical errors and alignment with the brand's voice.

This is the expected criteria for my final answer: A well-written blog post in markdown format, ready for publication, each section should have 2 or 3 paragraphs.

## This is the comprehensive content plan document on Artificial Intelligence:

**Task 1: Prioritize Latest Trends, Key Players, and Noteworthy News on Artificial Intelligence**

Latest Trends:
- advancements in Natural Language Processing (NLP)
- growth of Cloud-Based AI solutions
- increasing adoption of Explainable AI (XAI) technologies
- rising concerns around biased machine learning algorithms

Key Players:
- Microsoft Azure's AI capabilities for businesses
- Google DeepMind's latest breakthroughs in AlphaZero
- IBM's Watson Assistant for industry-specific applications
- NVIDIA's GPUs for deep learning and AI inference

Noteworthy News:
- The European Union announces plans to phase out single-use plastics, prompting companies to adopt more sustainable practices that can be implemented using AI.

### Task 2: Identify Target Audience

* Primary audience: tech-savvy professionals looking to enhance their business efficiency with AI tools
* Entrepreneurs seeking innovative solutions for their startups
* Small businesses and students exploring the feasibility of AI adoption

Secondary audience:
- Business leaders interested in understanding AI's potential impact on their operations
- Educators looking for research papers or articles on AI topics relevant to their curricula

Tertiary audience:
- Students exploring the concept, key players, and applications of Artificial Intelligence
- Anyone interested in learning about the latest discoveries and breakthroughs in AI research

### Task 3: Develop Detailed Content Outline

I. Introduction
- Brief overview of Artificial Intelligence and its increasing presence in various industries
- Explanation of AI's potential implications on business efficiency, sustainability, and innovation

II. Cloud-Based AI Solutions - Advancements in Natural Language Processing (NLP)
  * Applications: chatbots, voice assistants, sentiment analysis
* Relevant statistics:
  * Estimated global spending on cloud-based AI solutions within the next few years

III. Key Points on Emerging Trends - Growth of Explainable AI (XAI) Technologies
  - Solutions to common challenges in AI decision-making processes
* Key benefits of XAI Technologies:

IV. Business Implications and Adoption - The importance of adopting AI for business growth, efficiency, and scalability
    * Examples of successful startups leveraging AI across industries

V. Conclusion
- Recap of AI's growing presence across various sectors
- Recommendations on exploring Resources:
 1. Business cases focusing directly on AI adoption within their operations
 2. Trendsetters highlighting emerging trends with expert insight and analysis

VI. Key Points on Growing Adoption of Explainable AI (XAI) Technologies
  - Notable breakthroughs in areas like computer vision, reinforcement learning
* Understanding how to implement XAI for enhanced transparency and trust

VIII. Call to Action
- Encourage readers to explore resources and participate in online discussions about Artificial Intelligence.

### Task 4: Include SEO Keywords

* Primary keywords:
    + Artificial Intelligence
    + Cloud-Based AI Solutions
    + Explainable AI (XAI) Technologies
    + Natural Language Processing (NLP)
    + Google DeepMind
* Secondary keywords:
    + Trends in AI adoption
    + Key players in the AI industry
    + Notable breakthroughs in AI research
    + Business applications of AI

### Task 5: Identify Target Audience

Primary audience:
- Tech-savvy professionals seeking AI solutions for business efficiency
- Entrepreneurs looking to create innovative products and services with AI
- Small businesses and students targeting the feasibility of AI adoption

Secondary audience:
- Business leaders interested in understanding AI's potential impact on their operations
- Educators looking for research papers or articles on AI topics relevant to their curricula

Tertiary audience:
- Students exploring the concept, key players, and applications of Artificial Intelligence
- Anyone interested in learning about the latest discoveries and breakthroughs in AI research

### Task 6: Develop Audience Analysis

Demographic analysis:
* Primary audience demographics:
    + Age: 25-55 years old (tech-savvy professionals)
    + Location: Global offices and remote work arrangements
    + Education level: College graduates
    + Income level: Middle to high-income levels

Psychographic analysis:
* Characteristics of primary and secondary audiences:
    + Value innovation, efficiency, and scalability in AI solutions
    + Prioritize business implications over personal interests
    + Actively seek knowledge on the topic
    + Participate in online discussions related to AI adoption

Behavioral analysis:
* Users' behavior patterns when searching for and consuming content about Artificial Intelligence:
    + Primary audience: actively search for relevant resources, books, and articles
    + Secondary audience: following industry leaders and researchers on social media platforms
    + Tertiary audience: students exploring the concept of AI in educational projects

### Task 7: Develop Detailed SEO Keyword List

* Key terms:
  - Artificial Intelligence
  - Cloud-Based AI Solutions
  - Explainable AI (XAI) Technologies
  - Natural Language Processing (NLP)
  - Machine Learning

### Task 8: Develop Audience-Driven SEO Optimization

Based on audience analysis and behavior patterns, we will optimize our content to meet their needs and preferences.

* Primary audience:
    - Use clear titles with primary headings (e.g., "Emerging Trends in Cloud-Based AI Solutions")

* Secondary audience:
    - Display related section names to guide navigation
    - Break out extensive research and data into smaller sections

* Tertiary audience:
    - Use a clear call-to-action at the end of the article.

### Task 9: Develop Detailed Content Outline

I. Introduction
- Brief overview of Artificial Intelligence and its increasing presence in various industries
- Explanation of AI's potential implications on business efficiency, sustainability, and innovation

II. Cloud-Based AI Solutions - Advancements in Natural Language Processing (NLP)
    * Applications: chatbots, voice assistants, sentiment analysis
* Relevant statistics:
    - Estimated global spending on cloud-based AI solutions within the next few years

III. Key Points on Emerging Trends 
- Growth of Explainable AI (XAI) Technologies
  - Solutions to common challenges in AI decision-making processes
* Notable breakthroughs in areas like computer vision, reinforcement learning

IV. Business Implications and Adoption 
    * Examples of successful startups leveraging AI across industries
- Recap of AI's growing presence across various sectors
- Recommendations on exploring Resources:
 1. Business cases focusing directly on AI adoption within their operations
 2. Trendsetters highlighting emerging trends with expert insight and analysis

V. Key Points on Growing Adoption of Explainable AI (XAI) Technologies
    - Notable breakthroughs in areas like computer vision, reinforcement learning
* Understanding how to implement XAI for enhanced transparency and trust

VI. Call to Action 
- Encourage readers to explore resources and participate in online discussions about Artificial Intelligence.

VII. Conclusion
- Recap of AI's growing presence across various sectors
- Recommendations on further exploration and discussion about Artificial Intelligence.

VIII. Reference
- References cited throughout the content document

## Try it Yourself

- Pass in a topic of your choice and see what the agents come up with!

In [ ]:
topic = "YOUR TOPIC HERE"
result = crew.kickoff(inputs={"topic": topic})

In [ ]:
Markdown(result)

<a name='1'></a>
 ## Other Popular Models as LLM for your Agents

#### Hugging Face (HuggingFaceHub endpoint)

```Python
from langchain_community.llms import HuggingFaceHub

llm = HuggingFaceHub(
    repo_id="HuggingFaceH4/zephyr-7b-beta",
    huggingfacehub_api_token="<HF_TOKEN_HERE>",
    task="text-generation",
)

### you will pass "llm" to your agent function
```

#### Mistral API

```Python
OPENAI_API_KEY=your-mistral-api-key
OPENAI_API_BASE=https://api.mistral.ai/v1
OPENAI_MODEL_NAME="mistral-small"
```

#### Cohere

```Python
from langchain_community.chat_models import ChatCohere
# Initialize language model
os.environ["COHERE_API_KEY"] = "your-cohere-api-key"
llm = ChatCohere()

### you will pass "llm" to your agent function
```

### For using Llama locally with Ollama and more, checkout the crewAI documentation on [Connecting to any LLM](https://docs.crewai.com/how-to/LLM-Connections/).